# 3. Native PyTorch inference

Evaluate the checkpoint produced by notebook 2. The model returns the learned sequence from the entire image. It is a task-trained OCR reader, not an instruction-following VLM. Rotation correction is a separate optional pipeline.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "waterec_ocr").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository")
sys.path.insert(0, str(ROOT))
from waterec_ocr.waterec_config import load_config
from waterec_ocr.waterec_models import default_checkpoint
CONFIG_PATH = os.environ.get("WATEREC_CONFIG", "configs/dummy_alpr.yaml")
config = load_config(CONFIG_PATH)
checkpoint = default_checkpoint(config)
deployment = config["output_dir"] / "deployment"
examples = sorted((config["dataset_root"] / "val").glob("*.png"))
IMAGE_PATH = examples[0] if examples else config["dataset_root"] / "val/example.png"
def run(*args):
    return subprocess.run([sys.executable, "-m", "waterec_ocr.waterec_cli", *args, "--config", CONFIG_PATH], cwd=ROOT, check=True)
print("Profile:", CONFIG_PATH, "variant:", config["variant"], "patch budget:", config["tokens"])


In [ ]:
RUN_NATIVE = checkpoint.is_file()
if RUN_NATIVE:
    run("infer", "--image", str(IMAGE_PATH))
    run("evaluate", "--folder", str(config["dataset_root"] / "val"), "--output", str(config["output_dir"] / "evaluation_native"))
else:
    print("Train first; checkpoint missing:", checkpoint)


Exact text and edit similarity answer different questions. Price evaluation additionally compares the full numeric string, including its decimal value. Check difficult examples visually; do not interpret dummy accuracy as real-world capability.